<a href="https://colab.research.google.com/github/Aivon99/BigDataAndTextMiningProject/blob/main/src/eval/Task_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Downloading Libraries

In [1]:
!pip install -q transformers>=4.45.0 accelerate torch torchvision pillow scikit-learn tqdm

Importing Libraries

In [2]:
import os
import sys
import json
import subprocess
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from huggingface_hub import notebook_login
from datasets import load_dataset
from transformers import AutoModelForImageTextToText, AutoProcessor

Setting up environment

In [3]:
CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

if CONFIG["colab"]:
    repo_dir = Path(CONFIG["repo_dir"])
    if repo_dir.exists():
        subprocess.run(["rm", "-rf", str(repo_dir)], check=True)

    auth_url = "https://"
    repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

    result = subprocess.run(
        ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_dir)],
        capture_output=True, text=True
    )
    assert result.returncode == 0, f"Git clone failed: {result.stderr}"

    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))
else:
    repo_dir = Path(".").resolve()
    os.chdir(repo_dir)
    sys.path.insert(0, str(repo_dir))

REPO_ROOT = Path(".").resolve()
print("Setup Complete. REPO_ROOT:", REPO_ROOT)


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cpu


Dowloading dataset from HuggingFace repo

In [4]:
print("Verifying Autenthication to Hugging Face...")
notebook_login()

dataset_name = "bdatm-project/dataset_task1"
print(f"Downloading dataset '{dataset_name}'...")

dataset_task1 = load_dataset(dataset_name, token=True)

print("\nDataset loaded successfully!")
print(dataset_task1)
print("\nStructure sample of train split:")
print(dataset_task1["train"][0])

Verifying Autenthication to Hugging Face...


Resolving data files:   0%|          | 0/64 [00:00<?, ?it/s]


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image_count', 'image_files', 'image_size', 'patch_order', 'patch_size'],
        num_rows: 32
    })
    validation: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image_count', 'image_files', 'image_size', 'patch_order', 'patch_size'],
        num_rows: 4
    })
    test: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image_count', 'image_files', 'image_size', 'patch_order', 'patch_size'],
        num_rows: 4
    })
})

Structure sample of train split:
{'sample_id': 'sample_000000', 'puzzle_id': '2GDeK', 'task': 'task1', 'fen': 'r1b2rk1/ppRq3p/3p2p1/3PPp2/8/3B1NP1/2Q2K1P/8 b - - 1 30', 'prompt': 'You are a specialized model for chessboard understanding.\nYour goal is to extract the exact board state from the provided chessboard image.\nInput:\n- Board Ima

Loading Baseline Model

In [5]:
model_id = "Qwen/Qwen3.5-0.8B"
print(f"Loading model {model_id}...")

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
print("model and Processor loaded correctly!")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading model Qwen/Qwen3.5-0.8B...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model and Processor loaded correctly!


Test with baseline

In [11]:
import sys
from pathlib import Path
from PIL import Image
import torch

# Ensure the 'src' directory is in the system path for module imports
repo_root = Path("/content/BigDataAndTextMiningProject")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from data.generation import build_sample

# 1. Grab the first test sample directly from your loaded Hugging Face dataset variable
test_sample = dataset_task1["test"][0]

fen = test_sample["fen"]
task_prompt = test_sample["prompt"]
ground_truth_fen = test_sample["target"]
sample_id = test_sample["sample_id"]

# 2. Render the board image on the fly using the exact FEN from the dataset record
sample_output = build_sample(
    fen=fen,
    task="task1",
    sample_id=sample_id,
    image_size=512
)

board_image = sample_output["images"][0]

print(f"Sample ID: {sample_id}")
print(f"FEN: {fen}")
print(f"Prompt provided to the model:\n{task_prompt}\n")
print(f"Real FEN (Ground Truth): {ground_truth_fen}\n")

# 3. Prepare the multimodal input format for the model
chat_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": board_image},
            {"type": "text", "text": task_prompt},
        ]
    }
]

# 4. Apply the processor's chat template
formatted_text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

# 5. Tokenize inputs and move them to the GPU device
model_inputs = processor(
    text=[formatted_text],
    images=board_image,
    padding=True,
    return_tensors="pt"
).to(model.device)

# 6. Generate the zero-shot prediction
print("Generating zero-shot prediction...")
with torch.no_grad():
    output_token_ids = model.generate(**model_inputs, max_new_tokens=128)

# 7. Trim prompt tokens from the generated output
trimmed_output_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output_token_ids)
]
predicted_fen_string = processor.batch_decode(
    trimmed_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"Predicted FEN (Zero-Shot): {predicted_fen_string.strip()}")

ModuleNotFoundError: No module named 'cairosvg'